# 69. Prefix Caching Benchmark | 前缀缓存基准
**难度：** Hard | **环境：** CPU-first | **标签：** `推理优化`, `Prefix Cache`, `基准对比` | **目标人群：** 项目决策练习者

---

## 本节导读

多个请求如果共享相同的开头，前缀缓存可以让后续请求跳过已经完成的部分 Prefill。学习时要从请求分布出发，观察哪些前缀真的会重复、命中后节省了多少工作，以及缓存维护和失效带来了什么代价。本节形成缓存候选的专项证据并交给 72；它不负责选择完整服务组合。

**关键词：** `prefix cache`, `hit rate`, `TTFT`, `overhead`, `deployment`

---

## 前置阅读

**导语：** 进入本节前，先理解 KV Cache 如何保存请求状态，以及 PagedAttention、RadixAttention 如何组织这些状态。阅读时重点观察共享前缀如何被识别、命中和复用，再把这种复用与 TTFT、吞吐和缓存容量联系起来。
- [22. vLLM PagedAttention | vLLM PagedAttention](./22_vLLM_PagedAttention.ipynb)
- [24. Prefix Cache Matching and Reuse | Prefix Cache 匹配与复用](./24_Prefix_Cache_Matching_and_Reuse.ipynb)
- [34. SGLang RadixAttention | SGLang RadixAttention](./34_SGLang_RadixAttention.ipynb)



### Step 1：先定义可安全复用的缓存命名空间

多个请求共享开头时，前缀缓存把已经完成的 Prefill 状态按词元块保存下来。缓存键不能只包含文本：模型 revision、tokenizer、聊天模板、适配器、位置与注意力配置都会改变可复用状态。只有命名空间一致时，后续请求才从位置 0 开始累计连续完整块命中；未命中部分重新计算，容量不足时再按策略驱逐。

| 机制环节 | 要理解的状态变化 | 结果会影响什么 |
|:---|:---|:---|
| 共享前缀 | 请求是否有相同的开头 token | 可复用的候选范围 |
| 完整 block | 前缀按固定大小切分；尾块不参与命中 | 命中粒度与元数据开销 |
| 连续命中 | 只从第一个 block 开始累计命中 | reused tokens 与 Prefill 减少量 |
| 写入与驱逐 | 未命中 block 写入；容量满时移除旧 block | cache blocks、维护开销与后续命中 |
| 配置选择 | 比较 block、容量或驱逐策略的候选记录 | 是否值得进入 backend 复测 |

![前缀缓存 benchmark：请求复用、缓存策略与可观测证据](../docs/public/02_PyTorch_Algorithms/69_prefix_cache_benchmark_scope.svg)

### Step 2：固定复用分布，比较一个缓存变量

CPU 对照使用同一组已词元化请求，只改变块粒度、容量或驱逐策略。真实 backend 对照还要固定模型 revision、tokenizer、模板、适配器、请求顺序、生成长度、并发与测量过程，并确认后端日志确实命中前缀缓存。改变请求分布会直接改变命中机会，不能与缓存配置变化混在同一对照中。

| 对照对象 | 保持一致 | 唯一变化 | 必须满足的条件 |
|:---|:---|:---|:---|
| G0 cache off | 模型、tokenizer、shared-prefix workload、batch、并发 | 不复用前缀 | 作为 TTFT 与吞吐基线 |
| G1 prefix cache | 与 G0 相同 | 一组 block / capacity / eviction 配置 | 记录命中证据与缓存容量 |
| CPU 生命周期 | 同一 token 序列 | block_size 或 capacity_blocks | 尾块不命中；请求只处理一次 |

### Step 3：把复用收益和维护代价放进同一判断

命中率不是最终目标。缓存命名空间错配、输出不一致、运行失败或容量硬门槛失守时 `reject`；命中日志、原始请求、质量或显存证据缺失时 `tune`。证据完整后，再将复用词元、TTFT、吞吐、显存和驱逐代价放到同一结果中；只有命中确实减少 Prefill 工作并满足服务目标时才 `accept`。

| 证据组 | 字段 | 如何解释 |
|:---|:---|:---|
| 匹配与复用 | hit rate、reused tokens、prefill work reduction | 共享前缀是否真的减少了 Prefill 工作 |
| 性能 | TTFT、TPOT、E2E、throughput、P99 | 命中是否转化为端到端服务收益 |
| 维护成本 | peak memory、cache blocks、eviction、overhead | 高命中是否伴随不可接受的容量或维护压力 |
| 质量与失败 | 输出质量、OOM、unsupported、retest path | 不能把不可用或失败的候选当作收益 |
| 决策 | accept、tune、reject | 命中、TTFT 与维护成本必须共同判断 |

### Step 4：实现 CPU 缓存机制与候选选择

题目区使用分块与驱逐模型，重点判断缓存命名空间是否安全、后端证据是否证明真实复用，以及候选能否在延迟与容量门槛下进入部署复测。

| 任务与目的 | 已知条件 | 完成标准 | 测试验证 |
|:---|:---|:---|:---|
| TODO 1：核对缓存命名空间 | 模型、分词器、模板、适配器与位置配置 | 所有影响 KV 语义的身份一致 | 模板变化、适配器变化和位置配置变化 |
| TODO 2：建立缓存配对证据 | 无缓存基线、缓存候选与后端日志 | 硬失败、路径回退和关键缺证分别保留 | 伪命中、缺失显存和后端未命中 |
| TODO 3：形成采用状态 | 命中、TTFT、质量、容量与驱逐压力 | 正确复用且净收益达标才采用 | `accept`、`tune`、`reject` |


In [ ]:
from typing import Dict, List

In [ ]:
# 分块、驱逐和指标汇总构成机制底座；TODO 聚焦命名空间、后端证据和采用状态。
# CPU 结果验证证据关系，不替代真实缓存后端测量。
def complete_token_blocks(tokens: List[int], block_size: int) -> List[tuple[int, ...]]:
    """仅切出完整 token block；尾部不足一个 block 的 token 不参与缓存匹配。"""
    return [tuple(tokens[start:start + block_size]) for start in range(0, len(tokens) - block_size + 1, block_size)]


def count_continuous_prefix_hits(blocks, cache):
    """返回从 block 位置 0 开始连续命中的数量。

    blocks 是当前请求的完整 block 列表；cache 的 key 为 (block_index, block)。
    返回首个 miss 前的命中数，后续偶然命中不计入。
    """
    hit_blocks = 0
    for block_index, block in enumerate(blocks):
        if (block_index, block) not in cache:
            break
        hit_blocks += 1
    return hit_blocks


def write_blocks_with_lru(cache, blocks, clock, capacity_blocks):
    """写入当前请求的完整 block，并返回更新后的时钟与驱逐次数。

    已存在的 block 再次写入也要刷新访问时间；返回 (clock, eviction_count)。
    """
    eviction_count = 0
    for block_index, block in enumerate(blocks):
        clock += 1
        cache[(block_index, block)] = clock
        if len(cache) > capacity_blocks:
            oldest = min(cache, key=cache.get)
            del cache[oldest]
            eviction_count += 1
    return clock, eviction_count


def simulate_prefix_cache(token_sequences: List[List[int]], block_size: int = 4, capacity_blocks: int = 8) -> Dict[str, object]:
    """模拟完整 token block 的前缀复用和 LRU 驱逐；不测真实 backend 延迟。"""
    if block_size <= 0 or capacity_blocks <= 0:
        raise ValueError('block_size 和 capacity_blocks 必须为正数')
    if not isinstance(token_sequences, list) or not all(isinstance(tokens, list) for tokens in token_sequences):
        raise TypeError('token_sequences 必须是 list[list[int]]')
    cache, clock = {}, 0
    reused_tokens = total_prompt_tokens = hit_requests = eviction_count = 0
    for tokens in token_sequences:
        if not tokens:
            continue
        total_prompt_tokens += len(tokens)
        blocks = complete_token_blocks(tokens, block_size)
        hit_blocks = count_continuous_prefix_hits(blocks, cache)
        if hit_blocks:
            hit_requests += 1
        reused_tokens += hit_blocks * block_size
        clock, evicted = write_blocks_with_lru(cache, blocks, clock, capacity_blocks)
        eviction_count += evicted
    request_count = sum(bool(tokens) for tokens in token_sequences)
    return {'total_requests': request_count, 'total_prompt_tokens': total_prompt_tokens,
            'reused_tokens': reused_tokens, 'miss_tokens': total_prompt_tokens - reused_tokens,
            'hit_rate': hit_requests / request_count if request_count else 0.0,
            'token_reuse_rate': reused_tokens / total_prompt_tokens if total_prompt_tokens else 0.0,
            'prefill_work_reduction': reused_tokens / total_prompt_tokens if total_prompt_tokens else 0.0,
            'eviction_count': eviction_count, 'cache_size_blocks': len(cache)}


def summarize_prefix_cache(runs):
    """汇总同一 workload 的缓存运行记录；仅保留可比较的证据。"""
    if not runs:
        return {'run_count': 0, 'best_hit_rate_run': None, 'avg_hit_rate': 0.0, 'avg_ttft_ms': 0.0}
    required = {'name', 'workload_id', 'hit_rate', 'ttft_ms'}
    if any(not required.issubset(item) for item in runs):
        raise KeyError(f'每条 run 必须包含 {sorted(required)}')
    if len({item['workload_id'] for item in runs}) != 1:
        raise ValueError('缓存汇总必须使用同一 workload_id')
    best = max(runs, key=lambda item: item['hit_rate'])
    return {'run_count': len(runs), 'best_hit_rate_run': best['name'],
            'avg_hit_rate': sum(item['hit_rate'] for item in runs) / len(runs),
            'avg_ttft_ms': sum(item['ttft_ms'] for item in runs) / len(runs)}


def compare_prefix_cache_to_baseline(baseline, candidate):
    """计算同一 workload 下 candidate 相对 baseline 的复用、TTFT 与开销变化。"""
    required = {'workload_id', 'hit_rate', 'ttft_ms', 'overhead'}
    if not required.issubset(baseline) or not required.issubset(candidate):
        raise KeyError(f'比较记录必须包含 {sorted(required)}')
    if baseline['workload_id'] != candidate['workload_id']:
        raise ValueError('baseline 与 candidate 必须使用同一 workload_id')
    return {'hit_rate_gain': candidate['hit_rate'] - baseline['hit_rate'],
            'ttft_delta_ms': candidate['ttft_ms'] - baseline['ttft_ms'],
            'overhead_delta': candidate['overhead'] - baseline['overhead']}


def prefix_cache_rank_key(candidate):
    """返回一项证据合格缓存配置的升序排序 key。

    更大的 hit_rate_gain、更小的 ttft_delta_ms 和 overhead_delta 应排在前面；
    ttft_delta_ms 为负时表示 candidate 的 TTFT 更低。
    """
    summary = candidate['comparison']
    return (-summary['hit_rate_gain'], summary['ttft_delta_ms'], summary['overhead_delta'])


def select_prefix_cache_candidate(candidates):
    """排除无命中证据、失败或质量不合格候选后排序。"""
    eligible = [item for item in candidates if item['evidence_ok'] and item['quality_ok']]
    eligible.sort(key=prefix_cache_rank_key)
    return {'candidate_count': len(candidates), 'eligible_count': len(eligible),
            'selected': eligible[0] if eligible else None,
            'rejected_names': [item['name'] for item in candidates if item not in eligible]}


def validate_cache_namespace(baseline, candidate):
    """核对所有会改变 KV 语义的身份字段。"""
    namespace_fields = ('model_revision', 'tokenizer_id', 'chat_template', 'adapter_id', 'position_key')
    mismatches = [field for field in namespace_fields if baseline.get(field) != candidate.get(field)]
    # TODO 1：命名空间一致且候选只启用前缀缓存时，才允许复用比较。
    # namespace_ok：身份字段没有差异。
    # change_ok：基线关闭缓存，候选开启缓存。
    raise NotImplementedError('TODO 1：请核对缓存命名空间')


def build_prefix_cache_pair_evidence(baseline, candidate):
    """汇总命名空间、运行状态、后端命中和可观测服务指标。"""
    contract = validate_cache_namespace(baseline, candidate)
    failures = [name for name, record in (('baseline', baseline), ('candidate', candidate)) if record.get('failure')]
    required_metrics = ('hit_rate', 'ttft_ms', 'throughput_tps', 'output_complete', 'cache_capacity_blocks')
    missing = [field for field in required_metrics if candidate.get(field) is None]
    # TODO 2：契约或运行失败为硬失败；后端没有真实命中或关键字段缺失为缺证。
    raise NotImplementedError('TODO 2：请建立缓存配对证据')


def decide_prefix_cache_candidate(evidence, min_hit_rate, min_ttft_gain_ms, min_throughput_gain):
    """按输出完整性、直接命中与服务收益形成采用状态。"""
    # TODO 3：硬失败 reject；缺证或局部门槛未过 tune；全部通过 accept。
    # reuse_ok：命中率达标，且 TTFT 或吞吐至少一项形成净收益。
    # output_ok：两组请求均完整返回非空输出。
    raise NotImplementedError('TODO 3：请形成缓存候选状态')


def recommend_prefix_cache_run(baseline, candidate, min_hit_rate_gain, max_overhead):
    """将阈值判断转换为项目报告；决策字段不作为题目挖空。"""
    comparison = compare_prefix_cache_to_baseline(baseline, candidate)
    hit_rate_ok = comparison['hit_rate_gain'] >= min_hit_rate_gain
    ttft_ok = comparison['ttft_delta_ms'] < 0
    overhead_ok = comparison['overhead_delta'] <= max_overhead
    if hit_rate_ok and ttft_ok and overhead_ok:
        decision = 'accept'
    elif hit_rate_ok and ttft_ok:
        decision = 'tune'
    else:
        decision = 'reject'
    details = {'accept': ('命中率、延迟收益和维护开销都达标', 'promote_to_serving_eval'),
               'tune': ('命中率和延迟收益可用，但维护开销仍偏高', 'refine_chunk_or_eviction_policy'),
               'reject': ('命中率不足或延迟收益不明显', 'fallback_to_no_cache')}
    reason, next_action = details[decision]
    return {'decision': decision, 'reason': reason, 'next_action': next_action}

In [ ]:
# 本测试区验证缓存机制、候选配置排序和同 workload 的报告骨架。
def _prefix_cache_sequences():
    """构造共享前缀、部分命中与完全不相关的三条请求。"""
    return [[1, 2, 3, 4, 5, 6, 7, 8], [1, 2, 3, 4, 5, 6, 9, 10], [20, 21, 22, 23, 24, 25, 26, 27]]

def _cache_report(name, hit_rate, ttft_ms, overhead, evidence=True, quality=True):
    """构造同一 workload 下可用于缓存配置排序的显式指标记录。"""
    return {'name': name, 'workload_id': 'shared-prefix-v1', 'hit_rate': hit_rate, 'ttft_ms': ttft_ms,
            'overhead': overhead, 'evidence_ok': evidence, 'quality_ok': quality}

def test_continuous_block_hits():
    """验证连续前缀命中不能跳过首个 miss。"""
    cache = {(0, (1, 2)): 1, (1, (3, 4)): 2}
    assert count_continuous_prefix_hits([(1, 2), (3, 4), (5, 6)], cache) == 2
    assert count_continuous_prefix_hits([(9, 9), (3, 4)], cache) == 0

def test_lru_write_and_eviction():
    """验证容量上限、重新访问刷新时间和尾块不命中。"""
    cache = {(0, (1, 2)): 1}
    _, evicted = write_blocks_with_lru(cache, [(3, 4), (5, 6)], 1, 2)
    assert len(cache) == 2 and evicted == 1
    previous_clock = cache[(0, (3, 4))]
    refreshed_clock, no_extra_eviction = write_blocks_with_lru(cache, [(3, 4)], 3, 2)
    assert cache[(0, (3, 4))] == refreshed_clock > previous_clock and no_extra_eviction == 0
    simulated = simulate_prefix_cache(_prefix_cache_sequences(), block_size=2, capacity_blocks=4)
    assert simulated['reused_tokens'] == 6 and simulated['eviction_count'] > 0
    tail_case = simulate_prefix_cache([[1, 2, 3, 4, 5], [1, 2, 3, 4, 9]], block_size=4, capacity_blocks=8)
    assert tail_case['reused_tokens'] == 4
    for invalid in ({'block_size': 0}, {'capacity_blocks': 0}):
        try:
            simulate_prefix_cache([[1, 2]], **invalid)
        except ValueError:
            pass
        else:
            raise AssertionError('非法缓存参数应明确拒绝')

def test_candidate_ranking_and_contract():
    """验证证据和质量门槛先于缓存收益排序。"""
    baseline = _cache_report('off', 0., 130, 0.)
    small = _cache_report('small', .5, 100, .03)
    large = _cache_report('large', .7, 95, .08)
    candidates = [{'name': item['name'], 'evidence_ok': item['evidence_ok'], 'quality_ok': item['quality_ok'],
                   'comparison': compare_prefix_cache_to_baseline(baseline, item)} for item in [small, large]]
    assert select_prefix_cache_candidate(candidates)['selected']['name'] == 'large'
    blocked = {'name': 'blocked', 'evidence_ok': False, 'quality_ok': False,
               'comparison': {**candidates[1]['comparison'], 'hit_rate_gain': 99}}
    assert select_prefix_cache_candidate([blocked])['selected'] is None
    assert summarize_prefix_cache([baseline, small])['best_hit_rate_run'] == 'small'
    try:
        compare_prefix_cache_to_baseline(baseline, {**small, 'workload_id': 'other'})
    except ValueError:
        pass
    else:
        raise AssertionError('不同 workload 不应直接比较缓存收益')

def _cache_project_record(enabled, hit_rate, ttft_ms, throughput_tps, output_complete, cache_capacity_blocks):
    """构造命名空间一致、仅缓存开关不同的项目记录。"""
    return {'model_revision': 'commit', 'tokenizer_id': 'tokenizer', 'chat_template': 'template-v1',
            'adapter_id': 'none', 'position_key': 'rope-v1', 'prefix_cache_enabled': enabled,
            'backend_cache_hit_observed': enabled, 'hit_rate': hit_rate, 'ttft_ms': ttft_ms,
            'throughput_tps': throughput_tps, 'output_complete': output_complete,
            'cache_capacity_blocks': cache_capacity_blocks, 'failure': None}


def test_cache_project_contract_and_evidence():
    """验证命名空间、直接命中、输出完整性和服务收益门禁。"""
    baseline = _cache_project_record(False, 0.0, 130.0, 80.0, True, 37000)
    candidate = _cache_project_record(True, 0.65, 95.0, 84.0, True, 37000)
    assert validate_cache_namespace(baseline, candidate)['contract_ok']
    assert not validate_cache_namespace(baseline, {**candidate, 'chat_template': 'other'})['contract_ok']
    evidence = build_prefix_cache_pair_evidence(baseline, candidate)
    assert evidence['hard_failures'] == [] and evidence['missing_evidence'] == []
    assert decide_prefix_cache_candidate(evidence, 0.5, 20.0, 2.0)['decision'] == 'accept'
    no_path = build_prefix_cache_pair_evidence(baseline, {**candidate, 'backend_cache_hit_observed': False})
    assert decide_prefix_cache_candidate(no_path, 0.5, 20.0, 2.0)['decision'] == 'tune'
    bad_output = build_prefix_cache_pair_evidence(baseline, {**candidate, 'output_complete': False})
    assert decide_prefix_cache_candidate(bad_output, 0.5, 20.0, 2.0)['decision'] == 'reject'


def test_prefix_cache_benchmark_template():
    """运行本节全部 CPU 缓存机制测试。"""
    test_continuous_block_hits()
    test_lru_write_and_eviction()
    test_candidate_ranking_and_contract()
    test_cache_project_contract_and_evidence()
    print('测试通过：缓存命名空间、后端证据与采用状态均通过。')

try:
    test_prefix_cache_benchmark_template()
except NotImplementedError:
    print('请先完成 TODO 部分的代码！')
    raise

## 参考代码与解析


In [ ]:
# 分块、驱逐和指标汇总构成机制底座；TODO 聚焦命名空间、后端证据和采用状态。
# CPU 结果验证证据关系，不替代真实缓存后端测量。
def complete_token_blocks(tokens: List[int], block_size: int) -> List[tuple[int, ...]]:
    """仅切出完整 token block；尾部不足一个 block 的 token 不参与缓存匹配。"""
    return [tuple(tokens[start:start + block_size]) for start in range(0, len(tokens) - block_size + 1, block_size)]


def count_continuous_prefix_hits(blocks, cache):
    """返回从 block 位置 0 开始连续命中的数量。"""
    # 机制：位置和 token 内容共同构成 key，首个未命中后停止。
    hit_blocks = 0
    for block_index, block in enumerate(blocks):
        if (block_index, block) not in cache:
            break
        hit_blocks += 1
    return hit_blocks


def write_blocks_with_lru(cache, blocks, clock, capacity_blocks):
    """写入当前请求的完整 block，并返回更新后的时钟与驱逐次数。"""
    # 机制：访问刷新时钟，容量超限时驱逐最久未访问块。
    eviction_count = 0
    for block_index, block in enumerate(blocks):
        clock += 1
        cache[(block_index, block)] = clock
        while len(cache) > capacity_blocks:
            oldest = min(cache, key=cache.get)
            del cache[oldest]
            eviction_count += 1
    return clock, eviction_count


def simulate_prefix_cache(token_sequences: List[List[int]], block_size: int = 4, capacity_blocks: int = 8) -> Dict[str, object]:
    """模拟完整 token block 的前缀复用和 LRU 驱逐；不测真实 backend 延迟。"""
    if block_size <= 0 or capacity_blocks <= 0:
        raise ValueError('block_size 和 capacity_blocks 必须为正数')
    if not isinstance(token_sequences, list) or not all(isinstance(tokens, list) for tokens in token_sequences):
        raise TypeError('token_sequences 必须是 list[list[int]]')
    cache, clock = {}, 0
    reused_tokens = total_prompt_tokens = hit_requests = eviction_count = 0
    for tokens in token_sequences:
        if not tokens:
            continue
        total_prompt_tokens += len(tokens)
        blocks = complete_token_blocks(tokens, block_size)
        hit_blocks = count_continuous_prefix_hits(blocks, cache)
        if hit_blocks:
            hit_requests += 1
        reused_tokens += hit_blocks * block_size
        clock, evicted = write_blocks_with_lru(cache, blocks, clock, capacity_blocks)
        eviction_count += evicted
    request_count = sum(bool(tokens) for tokens in token_sequences)
    return {'total_requests': request_count, 'total_prompt_tokens': total_prompt_tokens,
            'reused_tokens': reused_tokens, 'miss_tokens': total_prompt_tokens - reused_tokens,
            'hit_rate': hit_requests / request_count if request_count else 0.0,
            'token_reuse_rate': reused_tokens / total_prompt_tokens if total_prompt_tokens else 0.0,
            'prefill_work_reduction': reused_tokens / total_prompt_tokens if total_prompt_tokens else 0.0,
            'eviction_count': eviction_count, 'cache_size_blocks': len(cache)}


def summarize_prefix_cache(runs):
    """汇总同一 workload 的缓存运行记录；仅保留可比较的证据。"""
    if not runs:
        return {'run_count': 0, 'best_hit_rate_run': None, 'avg_hit_rate': 0.0, 'avg_ttft_ms': 0.0}
    required = {'name', 'workload_id', 'hit_rate', 'ttft_ms'}
    if any(not required.issubset(item) for item in runs):
        raise KeyError(f'每条 run 必须包含 {sorted(required)}')
    if len({item['workload_id'] for item in runs}) != 1:
        raise ValueError('缓存汇总必须使用同一 workload_id')
    best = max(runs, key=lambda item: item['hit_rate'])
    return {'run_count': len(runs), 'best_hit_rate_run': best['name'],
            'avg_hit_rate': sum(item['hit_rate'] for item in runs) / len(runs),
            'avg_ttft_ms': sum(item['ttft_ms'] for item in runs) / len(runs)}


def compare_prefix_cache_to_baseline(baseline, candidate):
    """计算同一 workload 下 candidate 相对 baseline 的复用、TTFT 与开销变化。"""
    required = {'workload_id', 'hit_rate', 'ttft_ms', 'overhead'}
    if not required.issubset(baseline) or not required.issubset(candidate):
        raise KeyError(f'比较记录必须包含 {sorted(required)}')
    if baseline['workload_id'] != candidate['workload_id']:
        raise ValueError('baseline 与 candidate 必须使用同一 workload_id')
    return {'hit_rate_gain': candidate['hit_rate'] - baseline['hit_rate'],
            'ttft_delta_ms': candidate['ttft_ms'] - baseline['ttft_ms'],
            'overhead_delta': candidate['overhead'] - baseline['overhead']}


def prefix_cache_rank_key(candidate):
    """返回一项证据合格缓存配置的升序排序 key。"""
    summary = candidate['comparison']
    # 排序依据：更多复用、更低 TTFT 和更低维护开销优先。
    return (-summary['hit_rate_gain'], summary['ttft_delta_ms'], summary['overhead_delta'])


def select_prefix_cache_candidate(candidates):
    """排除无命中证据、失败或质量不合格候选后排序。"""
    eligible = [item for item in candidates if item['evidence_ok'] and item['quality_ok']]
    eligible.sort(key=prefix_cache_rank_key)
    return {'candidate_count': len(candidates), 'eligible_count': len(eligible),
            'selected': eligible[0] if eligible else None,
            'rejected_names': [item['name'] for item in candidates if item not in eligible]}


def validate_cache_namespace(baseline, candidate):
    """核对所有会改变 KV 语义的身份字段。"""
    namespace_fields = ('model_revision', 'tokenizer_id', 'chat_template', 'adapter_id', 'position_key')
    mismatches = [field for field in namespace_fields if baseline.get(field) != candidate.get(field)]
    # TODO 1：命名空间必须一致，唯一变化是是否启用前缀缓存。
    namespace_ok = not mismatches
    change_ok = baseline.get('prefix_cache_enabled') is False and candidate.get('prefix_cache_enabled') is True
    issues = list(mismatches) + ([] if change_ok else ['prefix_cache_enabled'])
    return {'contract_ok': namespace_ok and change_ok, 'issues': issues}


def build_prefix_cache_pair_evidence(baseline, candidate):
    """汇总命名空间、运行状态、后端命中和可观测服务指标。"""
    contract = validate_cache_namespace(baseline, candidate)
    failures = [name for name, record in (('baseline', baseline), ('candidate', candidate)) if record.get('failure')]
    required_metrics = ('hit_rate', 'ttft_ms', 'throughput_tps', 'output_complete', 'cache_capacity_blocks')
    missing = [field for field in required_metrics if candidate.get(field) is None]
    hard_failures = list(contract['issues']) + failures
    missing_evidence = list(missing)
    if not candidate.get('backend_cache_hit_observed'):
        missing_evidence.append('backend_cache_hit')
    return {'hard_failures': hard_failures, 'missing_evidence': missing_evidence,
            'hit_rate': candidate.get('hit_rate', 0.0),
            'ttft_gain_ms': baseline.get('ttft_ms', 0.0) - candidate.get('ttft_ms', 0.0),
            'throughput_gain': candidate.get('throughput_tps', 0.0) - baseline.get('throughput_tps', 0.0),
            'output_ok': baseline.get('output_complete') is True and candidate.get('output_complete') is True,
            'cache_capacity_blocks': candidate.get('cache_capacity_blocks')}


def decide_prefix_cache_candidate(evidence, min_hit_rate, min_ttft_gain_ms, min_throughput_gain):
    """按输出完整性、直接命中与服务收益形成采用状态。"""
    if evidence['hard_failures']:
        return {'decision': 'reject', 'next_action': 'repair_namespace_or_backend'}
    if evidence['missing_evidence']:
        return {'decision': 'tune', 'next_action': 'collect_cache_execution_evidence'}
    if not evidence['output_ok']:
        return {'decision': 'reject', 'next_action': 'inspect_request_outputs'}
    reuse_ok = (evidence['hit_rate'] >= min_hit_rate
                and (evidence['ttft_gain_ms'] >= min_ttft_gain_ms
                     or evidence['throughput_gain'] >= min_throughput_gain))
    if reuse_ok:
        return {'decision': 'accept', 'next_action': 'promote_fixed_workload_result'}
    return {'decision': 'tune', 'next_action': 'increase_shared_prefix_or_retest'}


def recommend_prefix_cache_run(baseline, candidate, min_hit_rate_gain, max_overhead):
    """将阈值判断转换为项目报告；决策字段不作为题目挖空。"""
    comparison = compare_prefix_cache_to_baseline(baseline, candidate)
    hit_rate_ok = comparison['hit_rate_gain'] >= min_hit_rate_gain
    ttft_ok = comparison['ttft_delta_ms'] < 0
    overhead_ok = comparison['overhead_delta'] <= max_overhead
    if hit_rate_ok and ttft_ok and overhead_ok:
        decision = 'accept'
    elif hit_rate_ok and ttft_ok:
        decision = 'tune'
    else:
        decision = 'reject'
    details = {'accept': ('命中率、延迟收益和维护开销都达标', 'promote_to_serving_eval'),
               'tune': ('命中率和延迟收益可用，但维护开销仍偏高', 'refine_chunk_or_eviction_policy'),
               'reject': ('命中率不足或延迟收益不明显', 'fallback_to_no_cache')}
    reason, next_action = details[decision]
    return {'decision': decision, 'reason': reason, 'next_action': next_action}

### 解析

分块和驱逐代码解释缓存内部状态；三个 TODO 负责项目级安全性、真实命中证据和采用判断。

#### TODO 1：核对缓存命名空间

模型版本、分词器、聊天模板、适配器和位置配置都会改变 KV 语义。任一身份变化都必须阻止复用比较。

#### TODO 2：建立缓存配对证据

契约错误和后端失败属于硬失败；缺少直接命中、缓存容量或请求结果属于缺证。启动开关不能代替执行时命中证据。

#### TODO 3：形成缓存候选状态

固定 workload 下，直接命中、输出完整和 TTFT/吞吐收益同时成立才采用。驱逐与容量压力仍由后续压力矩阵验证，不从本实验推断。

### Step 5（可选）：GPU 与 backend 实验——真实 prefix cache 对照

使用同一共享前缀请求分别运行 cache off、prefix cache 和单变量扩展配置；每次运行保存分组 JSON，再从 manifest 读取证据并形成项目决策。


| CPU 已验证 | 真实环境补充 | 本页输出 |
| 共享前缀、复用量与缓存代价 | cache off / prefix cache 的服务指标及可用命中证据 | 两组原始 JSON 与缓存候选决策 |


#### 5.1 环境与固定工作负载

本次实验回答一个具体问题：在同一组共享前缀请求上，开启 prefix cache 是否带来可观测收益。先记录实验契约，再让 G0 与 G1 只改变 cache policy；模型、backend、请求分布、生成长度、并发和测量流程保持一致。

| 实验要素 | 本轮契约 | G0 / G1 如何保持一致 | 证据输出 |
|:---|:---|:---|:---|
| 实验对象 | G0 cache off；G1 prefix cache | 使用同一模型 revision、backend 和服务版本 | 两组可比较结果 |
| 固定 workload | shared-prefix 请求文件、生成长度、batch / concurrency | 使用同一请求顺序和 token 化结果 | 统一 workload 记录 |
| 测量流程 | warmup、重复次数、超时和输出长度 | 两组使用相同流程 | 可复查的 JSON 结果 |
| 唯一变量 | cache policy | G0 关闭，G1 只开启 prefix cache | 命中、延迟、吞吐和资源对照 |

![前缀缓存 GPU/backend 对照实验契约](../docs/public/02_PyTorch_Algorithms/69_prefix_cache_gpu_contract.svg)


使用 `INF-VLLM-011-CU128` Profile，硬件下限为单张 12 GB NVIDIA GPU；环境创建见[推理优化 Quickstart](../topic_discussion/inference_optimization/quickstart.md#inf-vllm-quickstart)。

In [ ]:
# 5.1：唯一可编辑配置。先改这一处，再运行 5.2–5.6。
RUN_REAL_BACKEND = False  # 默认值 False：改为 True 才启动真实 G0/G1 服务对照。
RUN_LABEL = 'rtx5070ti_prefix_cache_v4'  # 正式复测改为唯一标签；旧标签的 JSON 保留为历史证据。
SEMANTIC_ID = 'INF-PREFIX-CACHE'
SCHEMA_VERSION = 'inference-benchmark/v1'
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # G0/G1 使用同一模型与 revision。
WORKLOAD = 'benchmarks/workloads/prefix_reuse_long.jsonl'  # 长共享前缀；两组必须使用同一冻结请求集。
EXECUTION_ORDER = 'baseline_then_candidate'  # 可选 baseline_then_candidate / candidate_then_baseline；反向复测需新标签。
NUM_PROMPTS = 30  # 按固定顺序循环 workload，不改变唯一请求集合。
CONCURRENCY = 1  # 先隔离 prefix cache；并发扫描属于 70，不在本页混入。
REPEATS = 3  # 避免把单次服务波动当作缓存结论。
WARMUP = 5  # 预热不计入正式统计；G1 的预热也会建立可复用条目。
MAX_TOKENS = 64  # 两组保持相同的生成长度。
BACKEND = 'vllm'  # 改变 backend 应形成新的专项对照。
DTYPE = 'bfloat16'  # 记录 backend 实际选择的 dtype；两组必须一致。

MAX_MODEL_LEN = 2048  # G0/G1 固定相同 KV 容量上限。
GPU_MEMORY_UTILIZATION = 0.80
ENFORCE_EAGER = True
RUN_ID = RUN_LABEL  # 派生内部标识；不单独修改。
MANIFEST_PATH = f'benchmarks/results/69_manifest_{RUN_ID}.json'  # 对照清单。
CANONICAL_GROUPS = (
    {'name': 'g0_cache_off', 'role': 'baseline', 'cache_policy': 'off', 'enabled': False},
    {'name': 'g1_prefix_cache', 'role': 'candidate', 'cache_policy': 'prefix_cache', 'enabled': True},
)
GROUPS = CANONICAL_GROUPS if EXECUTION_ORDER == 'baseline_then_candidate' else tuple(reversed(CANONICAL_GROUPS))

#### 5.2 环境启动检查

确认共享前缀 workload 可被读取，并记录 backend、GPU 与缓存开关。预检结果用于解释后续启动失败或缺少命中指标的原因。

| 检查项 | 需要确认的内容 | 记录用途 |
|:---|:---|:---|
| 模型与 backend | G0/G1 使用同一模型 revision 与服务版本 | 保持对照条件一致 |
| GPU 与运行时 | GPU、驱动、CUDA、PyTorch、dtype | 环境变更后重新采集 |
| workload 与开关 | 请求文件、端口、cache policy | 确认共享前缀请求和策略可加载 |
| 命中证据 | backend metrics、日志或专用接口 | 不能由 TTFT 或吞吐反推命中率 |


In [ ]:
# 5.2：只读取 5.1 配置并检查运行条件，不启动服务。
from pathlib import Path
from tools.inference_project_runtime import locate_repo_root
import importlib.util
import torch

REPO_ROOT = locate_repo_root()  # 运行基础设施：将 5.1 的仓库相对路径锚定到项目根目录。
WORKLOAD_FILE = REPO_ROOT / WORKLOAD
MANIFEST_FILE = REPO_ROOT / MANIFEST_PATH
preflight = {
    'workload_exists': WORKLOAD_FILE.is_file(), 'backend': BACKEND,
    'cuda_available': torch.cuda.is_available(),
    'vllm_on_current_kernel': importlib.util.find_spec('vllm') is not None,
}
print('prefix cache preflight:', preflight)

#### 5.3 候选与比较契约检查

本单元只验证 5.1 声明的请求集、两组缓存策略和输出路径可比较；不再定义实验参数。


In [ ]:
from pathlib import Path
from tools.backend_runtime import start_vllm as loaded_start_vllm

if '--no-enable-prefix-caching' not in loaded_start_vllm.__code__.co_consts:
    raise RuntimeError(
        '当前 Kernel 仍加载旧版 backend_runtime，无法构造 cache-off 基线；'
        '请重启 Kernel 后从 5.1 重新运行。'
    )
if not WORKLOAD_FILE.is_file():
    raise FileNotFoundError(f'找不到共享前缀 workload：{WORKLOAD_FILE}')
if EXECUTION_ORDER not in {'baseline_then_candidate', 'candidate_then_baseline'}:
    raise ValueError('EXECUTION_ORDER 只能是 baseline_then_candidate 或 candidate_then_baseline')
if {group['name'] for group in GROUPS} != {'g0_cache_off', 'g1_prefix_cache'}:
    raise ValueError('本页必须保留 cache off 与 prefix cache 两个受控组')
if NUM_PROMPTS <= 0 or WARMUP < 0 or REPEATS <= 0:
    raise ValueError('num_prompts、repeats 必须为正数，warmup 不能为负数')
project_config = {
    'model': MODEL_ID, 'backend': BACKEND, 'dtype': DTYPE, 'workload': WORKLOAD,
    'generated_tokens': MAX_TOKENS, 'num_prompts': NUM_PROMPTS,
    'concurrency': CONCURRENCY, 'warmup': WARMUP, 'repeats': REPEATS,
    'execution_order': EXECUTION_ORDER,
    'groups': [item['name'] for item in GROUPS],
}
print(project_config)

#### 5.4 执行实验并保存原始 JSON

按 5.1 中声明的顺序运行 G0 与 G1。每一组完成 warmup 后再测量；单组报告和本次 manifest 只记录原始配置、请求结果、缓存观测与失败信息。结论由 5.6 单独生成，不能回写这些原始文件。

| 分支 | 执行内容 | 原始产物 |
| --- | --- | --- |
| G0 | cache off，固定共享前缀 workload | `69_g0_cache_off_...json` |
| G1 | prefix cache，其他条件不变 | `69_g1_prefix_cache_...json` |
| manifest | 汇总本次两组、结果路径与运行状态 | `69_manifest_...json` |

In [ ]:
# 5.4：执行 G0/G1；在服务仍运行时登记 prefix-cache 观测，再保存分组 JSON。
from pathlib import Path
import json
import re
import urllib.request
from tools.inference_project_runtime import runtime_snapshot, start_optional_vllm, stop_optional_vllm, run_backend_benchmark
from tools.project_runtime import save_json_record_once


def observe_prefix_cache(base_url, log_handle):
    """从当前 vLLM 的 /metrics 与服务日志记录缓存命中证据。"""
    observation = {
        'hit_rate': None,
        'cache_queries': None,
        'cache_hits': None,
        'cache_capacity_blocks': None,
        'hit_rate_evidence': None,
        'metrics_source': f'{base_url}/metrics',
        'prefix_cache_enabled': None,
        'prefix_cache_metric_series': [],
        'log_path': getattr(log_handle, 'name', None),
    }
    try:
        with urllib.request.urlopen(f'{base_url}/metrics', timeout=10) as response:
            metric_text = response.read().decode('utf-8', errors='replace')
        series = []
        for line in metric_text.splitlines():
            normalized = line.lower()
            if 'prefix' in normalized and 'cache' in normalized and not line.startswith('#'):
                series.append(line)
        observation['prefix_cache_metric_series'] = series
        config_line = next((line for line in series if 'cache_config_info' in line), '')
        enabled = re.search(r'enable_prefix_caching=\"(True|False)\"', config_line)
        if enabled:
            observation['prefix_cache_enabled'] = enabled.group(1) == 'True'
        queries = re.search(r'prefix_cache_queries_total[^}]*}\s+([0-9.]+)', metric_text)
        hits = re.search(r'prefix_cache_hits_total[^}]*}\s+([0-9.]+)', metric_text)
        blocks = re.search(r'num_gpu_blocks=\"([0-9]+)\"', config_line)
        if queries and hits:
            observation['cache_queries'] = float(queries.group(1))
            observation['cache_hits'] = float(hits.group(1))
            if observation['cache_queries'] > 0:
                observation['hit_rate'] = round(observation['cache_hits'] / observation['cache_queries'], 6)
        if blocks:
            observation['cache_capacity_blocks'] = int(blocks.group(1))
        if series:
            observation['hit_rate_evidence'] = 'vllm_metrics_prefix_cache_series'
    except Exception as exc:
        observation['metrics_error'] = str(exc)

    try:
        log_handle.flush()
        log_text = Path(log_handle.name).read_text(encoding='utf-8', errors='replace')
        matches = re.findall(r'Prefix cache hit rate:\s*([0-9.]+)%', log_text)
        if matches:
            observation['hit_rate'] = round(float(matches[-1]) / 100.0, 6)
            observation['hit_rate_evidence'] = 'vllm_server_log_prefix_cache_hit_rate'
    except Exception as exc:
        observation['log_parse_error'] = str(exc)
    return observation


if RUN_REAL_BACKEND:
    final_paths = {group['name']: REPO_ROOT / f"benchmarks/results/69_{group['name']}_{BACKEND}_{RUN_ID}.json" for group in GROUPS}
    raw_paths = {group['name']: REPO_ROOT / f"benchmarks/results/69_{group['name']}_{BACKEND}_raw_{RUN_ID}.json" for group in GROUPS}
    collisions = [path for path in (*final_paths.values(), *raw_paths.values(), MANIFEST_FILE) if path.exists()]
    if collisions:
        raise FileExistsError(f'当前 RUN_LABEL 已有结果，请更换标签：{collisions}')
    reports = []
    for group in GROUPS:
        result_path = final_paths[group['name']]
        raw_result_path = raw_paths[group['name']]
        server = log_path = None
        try:
            server, log_path, port, selected_dtype, _ = start_optional_vllm(
                model_id=MODEL_ID, model_source='auto', dtype=DTYPE,
                served_model_name=MODEL_ID, enable_prefix_caching=group['enabled'],
                max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
                enforce_eager=ENFORCE_EAGER,
                log_path=REPO_ROOT / f"benchmarks/results/69_{group['name']}_{BACKEND}_{RUN_ID}.log",
            )
            base_url = f'http://127.0.0.1:{port}'
            report = run_backend_benchmark(
                project='69', base_url=base_url, model=MODEL_ID,
                label=f"{BACKEND}-{group['name']}", output=str(raw_result_path), workload=str(WORKLOAD_FILE),
                backend=BACKEND, batch=1, max_tokens=MAX_TOKENS, num_prompts=NUM_PROMPTS,
                concurrency=CONCURRENCY, warmup=WARMUP, repeats=REPEATS, dtype=selected_dtype,
                cache_policy=group['cache_policy'], include_requests=True,
            )
            report.update({'schema_version': SCHEMA_VERSION, 'semantic_id': SEMANTIC_ID, 'run_label': RUN_LABEL, 'run_mode': 'real_backend', 'raw_result_path': str(raw_result_path)})
            report['strategy_metrics'] = observe_prefix_cache(base_url, log_path)
            save_json_record_once(result_path, report)
            reports.append({'group': group, 'role': group['role'], 'report_path': str(result_path),
                            'status': report.get('status', 'ok'), 'quality': report.get('quality', {'status': 'not_evaluated'}),
                            'failure': report.get('failure'), 'evidence_level': 'real_backend_paired'})
        except Exception as error:
            failure_record = {'schema_version': SCHEMA_VERSION, 'semantic_id': SEMANTIC_ID, 'project': '69', 'run_label': RUN_LABEL, 'run_mode': 'real_backend', 'status': 'failed', 'group': group, 'failure': {'type': type(error).__name__, 'message': str(error)}}
            if not result_path.exists():
                save_json_record_once(result_path, failure_record)
            reports.append({'group': group, 'role': group['role'], 'report_path': str(result_path),
                            'status': 'failed', 'quality': {'status': 'failed'},
                            'failure': {'failure_reason': str(error), 'retest_path': 'recheck_backend_and_cache_policy'},
                            'evidence_level': 'backend_execution_failure'})
        finally:
            if server is not None:
                stop_optional_vllm(server, log_path)
    manifest = {'schema_version': 'inference-project-manifest/v1', 'semantic_id': SEMANTIC_ID, 'project': '69', 'run_label': RUN_LABEL, 'run_mode': 'real_backend',
        'experiment': 'prefix_cache_reuse', 'config': project_config, 'environment': runtime_snapshot(), 'groups': reports,
        'artifact': {'manifest_path': MANIFEST_PATH, 'group_result_paths': [item['report_path'] for item in reports]},
        'failure': [item for item in reports if item.get('status') != 'ok'] or None,
        'quality': {'status': 'not_evaluated'}, 'evidence_level': 'real_backend_paired',
        'decision': {'decision': 'tune', 'reason': '运行 5.5、5.6 汇总命中、质量和性能证据后再判断'},}
    output = MANIFEST_FILE
    save_json_record_once(output, manifest)
    print(f'对照清单已保存: {output}')
else:
    print('实验未启动：确认 5.1 条件后，将 RUN_REAL_BACKEND 改为 True。')

#### 5.5 读取原始结果并规整证据

逐条读取 manifest 引用的 G0/G1 JSON，确认两组共享模型、backend、dtype、workload、生成长度和并发。这里检查的是请求是否完整返回，不把“非空输出”表述为模型质量评测；语义质量需由独立评测集或任务判据提供。

| 组别 | 唯一变化 | 缓存机制证据 | 请求完成与性能 | 证据用途 |
| --- | --- | --- | --- | --- |
| G0 | cache off | 实际开关为关闭 | TTFT / TPOT / 吞吐 / 显存；成功与失败请求 | 可比较的基线 |
| G1 | prefix cache | 实际开关、hit rate、证据来源 | 与 G0 同口径；请求完成状态 | 判断复用是否带来净收益 |
| 失败记录 | unsupported / OOM / failed | 已采集字段 | failure reason、retest path | 不与成功运行混合平均 |

In [ ]:
# 5.5 读取分组 JSON，核对共同配置并规整命中、质量与性能证据。
import json
from pathlib import Path
from tools.inference_project_runtime import locate_repo_root

REPO_ROOT = locate_repo_root()
RESULT_CONFIG_FIELDS = ('model', 'backend', 'dtype', 'workload', 'generated_tokens', 'concurrency')


def scalar_metric(metrics, name, statistic='p50'):
    """读取标量指标或嵌套分位数。"""
    value = metrics.get(name)
    if isinstance(value, dict):
        return value.get(statistic)
    return value


def read_prefix_result(result_path):
    """读取单组报告，展平延迟分位数并保留可追溯的缓存观测来源。"""
    result_path = Path(result_path)
    if not result_path.is_absolute():
        result_path = REPO_ROOT / result_path
    if not result_path.exists():
        return {'path': str(result_path), 'status': 'missing'}
    raw = json.loads(result_path.read_text(encoding='utf-8'))
    normalized = raw.get('normalized_result') or raw
    config = raw.get('config') or normalized.get('config', {})
    raw_metrics = raw.get('metrics') or normalized.get('metrics', {})
    results = raw.get('results') or []
    expected = raw.get('workload', {}).get('requests')
    successful = raw_metrics.get('successful_requests')
    failed = raw_metrics.get('failed_requests')
    nonempty_outputs = all(bool(item.get('output_text', '')) for item in results if item.get('ok'))
    quality_status = 'passed' if expected == successful and failed == 0 and nonempty_outputs else 'not_evaluated'
    strategy = raw.get('strategy_metrics') or normalized.get('strategy_metrics', {})
    status = raw.get('status') or normalized.get('status')
    if status in {None, 'not_recorded'}:
        status = 'failed' if raw.get('failure') else 'ok'
    return {
        'path': str(result_path),
        'status': status,
        'config': {field: config.get(field) for field in RESULT_CONFIG_FIELDS},
        'metrics': {
            'ttft_p50_ms': scalar_metric(raw_metrics, 'ttft_ms', 'p50'),
            'ttft_p95_ms': scalar_metric(raw_metrics, 'ttft_ms', 'p95'),
            'tpot_p50_ms': scalar_metric(raw_metrics, 'tpot_ms', 'p50'),
            'tpot_p95_ms': scalar_metric(raw_metrics, 'tpot_ms', 'p95'),
            'throughput_tokens_per_s': raw_metrics.get('output_token_throughput_per_s'),
            'e2e_p95_ms': scalar_metric(raw_metrics, 'e2e_ms', 'p95'),
            'successful_requests': successful,
            'failed_requests': failed,
        },
        'cache_metrics': {field: strategy.get(field) for field in (
            'hit_rate', 'cache_queries', 'cache_hits', 'cache_capacity_blocks', 'hit_rate_evidence',
            'metrics_source', 'prefix_cache_enabled', 'prefix_cache_metric_series', 'log_path',
        )},
        'output_completeness': {'status': quality_status, 'expected_requests': expected,
                                'successful_requests': successful, 'nonempty_outputs': nonempty_outputs},
        'failure': raw.get('failure') or normalized.get('failure', {}),
    }


def summarize_prefix_pair(records):
    """汇总缓存候选相对基线的配对差值。"""
    indexed = {item.get('group'): item for item in records}
    g0, g1 = indexed.get('g0_cache_off'), indexed.get('g1_prefix_cache')
    if not g0 or not g1:
        return {}
    base, candidate = g0.get('metrics', {}), g1.get('metrics', {})
    def delta(name):
        left, right = base.get(name), candidate.get(name)
        return None if left is None or right is None else round(right - left, 4)
    return {
        'ttft_p50_delta_ms': delta('ttft_p50_ms'),
        'ttft_p95_delta_ms': delta('ttft_p95_ms'),
        'tpot_p50_delta_ms': delta('tpot_p50_ms'),
        'throughput_delta_tokens_per_s': delta('throughput_tokens_per_s'),
        'e2e_p95_delta_ms': delta('e2e_p95_ms'),
    }


def read_prefix_manifest(manifest_path):
    """读取 manifest 及分组结果，并检查 G0/G1 是否具有可比性。"""
    manifest_path = Path(manifest_path)
    if not manifest_path.is_absolute():
        manifest_path = REPO_ROOT / manifest_path
    if not manifest_path.exists():
        return {'status': 'not_run', 'manifest': str(manifest_path), 'results': []}
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    groups = manifest.get('groups', [])
    results = []
    for group in groups:
        item = read_prefix_result(group.get('report_path', ''))
        item['group'] = group.get('group', {}).get('name')
        item['role'] = group.get('role')
        if item['status'] == 'missing' and group.get('status') == 'failed':
            item.update({'status': 'failed', 'failure': group.get('failure', {})})
        results.append(item)
    readable = [item for item in results if item.get('status') not in {'missing', 'failed'}]
    signatures = {tuple(item['config'].get(field) for field in RESULT_CONFIG_FIELDS) for item in readable}
    return {
        'status': 'ok', 'manifest': str(manifest_path), 'results': results,
        'shared_config_ok': len(readable) == 2 and len(signatures) == 1,
        'paired_summary': summarize_prefix_pair(results),
        'output_completeness': {item.get('group'): item.get('output_completeness') for item in results},
        'failure': manifest.get('failure'),
    }

prefix_evidence = read_prefix_manifest(globals().get('MANIFEST_PATH', 'benchmarks/results/69_manifest_smoke.json'))
print(json.dumps(prefix_evidence, ensure_ascii=False, indent=2))

#### 5.6 形成派生决策

先确认比较口径和直接命中证据，再查看请求完成状态、TTFT、吞吐与容量观察。5.6 只读取 5.5 的证据并生成派生决策；不修改 G0/G1 原始 JSON 或 manifest。

| 观察到的结果 | 决策 | 下一步 |
| --- | --- | --- |
| 命中证据完整、请求完整，TTFT 或吞吐改善，容量开销可接受 | `accept` | 扩展请求分布并做回归测试 |
| 结果不全、命中证据缺失，或性能信号不一致 | `tune` | 补充指标或调整 block、容量、驱逐策略 |
| unsupported、OOM、请求失败或端到端性能同时退化 | `reject` | 保留失败记录，回退 cache off 或更换候选 |

In [ ]:
# 5.6 只根据 5.5 的证据给出派生决策；不修改 5.4 写出的原始 JSON 或 manifest。
def decide_prefix_cache_evidence(evidence):
    """优先检查失败与可比性，再以命中率、尾延迟和吞吐判断当前 workload。"""
    if evidence.get('status') != 'ok':
        return {'decision': 'tune', 'reason': '尚未获得可读取的分组结果'}
    if evidence.get('failure'):
        return {'decision': 'reject', 'reason': '存在 backend 失败记录'}
    records = {item.get('group'): item for item in evidence.get('results', [])}
    g0, g1 = records.get('g0_cache_off'), records.get('g1_prefix_cache')
    if not g0 or not g1 or not evidence.get('shared_config_ok'):
        return {'decision': 'tune', 'reason': 'G0/G1 结果或共同配置尚未完整'}
    if any(item.get('output_completeness', {}).get('status') != 'passed' for item in (g0, g1)):
        return {'decision': 'reject', 'reason': 'G0 或 G1 请求未完整返回，不能解释缓存性能'}
    g0_cache, g1_cache = g0.get('cache_metrics', {}), g1.get('cache_metrics', {})
    if g0_cache.get('prefix_cache_enabled') is not False or g1_cache.get('prefix_cache_enabled') is not True:
        return {'decision': 'reject', 'reason': 'G0/G1 的实际 prefix-cache 开关未形成关闭/开启对照'}
    if (g1_cache.get('hit_rate') is None or not g1_cache.get('hit_rate_evidence')
            or g1_cache.get('cache_capacity_blocks') is None):
        return {'decision': 'tune', 'reason': '缺少来自 backend 指标或服务日志的直接命中证据'}
    base, candidate = g0.get('metrics', {}), g1.get('metrics', {})
    ttft95_0, ttft95_1 = base.get('ttft_p95_ms'), candidate.get('ttft_p95_ms')
    throughput_0, throughput_1 = base.get('throughput_tokens_per_s'), candidate.get('throughput_tokens_per_s')
    if None in {ttft95_0, ttft95_1, throughput_0, throughput_1}:
        return {'decision': 'tune', 'reason': 'TTFT P95 或输出吞吐字段不完整'}
    if ttft95_1 < ttft95_0 and throughput_1 > throughput_0:
        return {
            'decision': 'accept',
            'reason': '当前共享前缀 workload 中，直接命中证据、请求完整性、TTFT P95 与吞吐对照均通过',
            'scope': '单卡、单并发、固定共享前缀 workload；输出完整性已检查，但未覆盖高并发、驱逐压力或语义质量评测',
        }
    if ttft95_1 > ttft95_0 and throughput_1 < throughput_0:
        return {'decision': 'reject', 'reason': '缓存配置使 TTFT P95 与吞吐同时退化'}
    return {'decision': 'tune', 'reason': '命中存在但性能信号不一致；升级共享前缀长度或并发后复测'}

prefix_cache_decision = decide_prefix_cache_evidence(globals().get('prefix_evidence', {}))
print({'decision': prefix_cache_decision, 'evidence_source': '5.5_read_only'})

## 相关阅读

**项目与开源实现**

完成前缀缓存 benchmark 后，可用 70 继续检查缓存命中与请求调度之间的关系，并保持相同的 workload 记录方式。下面的实现文档和论文用于继续追踪真实系统中的缓存组织、命中和显存管理。
- [70. Serving Scheduler Benchmark | 推理服务调度基准](./70_Serving_Scheduler_Benchmark.ipynb)
- [Automatic Prefix Caching（vLLM 开源实现文档）](https://docs.vllm.ai/en/latest/design/prefix_caching/)
- [Radix Cache（SGLang 开源实现）](https://github.com/sgl-project/sglang/blob/main/python/sglang/srt/mem_cache/radix_cache.py)
- [Efficient Memory Management for Large Language Model Serving with PagedAttention（论文）](https://arxiv.org/abs/2309.06180)

> 注意：OpenAI-compatible API 通常不会直接返回 prefix-cache hit rate。真实 backend 实验可以用共享前缀 workload 比较 TTFT 和吞吐，但不能仅凭延迟变化反推出命中率；命中率必须来自 backend metrics、日志或专用观测接口。
